---
### Konteks Skenario:
Tim data engineering platform e-commerce meminta kita membangun **pipeline ETL produksi pertama** yang menggabungkan tiga sumber data sekaligus:
1. **Data Transaksi (`tugas6_transaksi.csv`)**: 5.000 baris rekaman transaksi penjualan e-commerce berformat CSV.
2. **Data Produk (`tugas6_produk.json`)**: 30 data master produk berformat JSON Lines (NDJSON).
3. **Data Ulasan Pelanggan (`tugas6_ulasan.csv`)**: 3.500 baris ulasan rating kepuasan pelanggan berformat CSV (merepresentasikan kondisi nyata di mana tidak semua transaksi memiliki ulasan).


In [1]:
import os
import sys
import json
import numpy as np
import pandas as pd

# Memastikan driver dan worker Spark menggunakan interpreter Python yang sama
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

# Menetapkan random seed agar data konsisten dan deterministik
np.random.seed(33)

# 1. Sumber 1: Transaksi (CSV) — 5000 baris
n_trx = 5000
kategori_list = ["Elektronik", "Fashion", "Makanan", "Rumah Tangga", "Kesehatan"]
df_trx_tugas6 = pd.DataFrame({
    "order_id": [f"TX{i}" for i in range(n_trx)],
    "product_id": np.random.randint(1, 31, size=n_trx),
    "unit_terjual": np.random.randint(1, 8, size=n_trx),
    "tanggal": np.random.choice(pd.date_range("2026-10-01", "2026-10-31"), size=n_trx).astype(str),
})
df_trx_tugas6.to_csv("tugas6_transaksi.csv", index=False)

# 2. Sumber 2: Data produk (JSON Lines) — 30 produk
produk = [
    {
        "product_id": i,
        "nama_produk": f"Produk-{i}",
        "kategori": np.random.choice(kategori_list),
        "harga": int(np.random.choice([25000, 50000, 75000, 100000, 150000, 250000]))
    }
    for i in range(1, 31)
]
with open("tugas6_produk.json", "w") as f:
    for p in produk:
        f.write(json.dumps(p) + "\n")

# 3. Sumber 3: Data ulasan (CSV) — 3500 ulasan (kondisi riil: sebagian transaksi tidak memiliki review)
n_review = 3500
df_review = pd.DataFrame({
    "order_id": np.random.choice(df_trx_tugas6["order_id"], size=n_review, replace=False),
    "rating": np.random.randint(1, 6, size=n_review),
})
df_review.to_csv("tugas6_ulasan.csv", index=False)

print("Tiga sumber data berhasil dibuat:")
print(f"- tugas6_transaksi.csv : {len(df_trx_tugas6)} baris")
print(f"- tugas6_produk.json   : {len(produk)} baris")
print(f"- tugas6_ulasan.csv    : {len(df_review)} baris (tidak seluruh transaksi memiliki ulasan)")

Tiga sumber data berhasil dibuat:
- tugas6_transaksi.csv : 5000 baris
- tugas6_produk.json   : 30 baris
- tugas6_ulasan.csv    : 3500 baris (tidak seluruh transaksi memiliki ulasan)


---
## Inisialisasi SparkSession

Membuat sesi Spark (`SparkSession`) dengan konfigurasi mode lokal paralel (`local[*]`) untuk memproses beban kerja ETL terdistribusi.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, count, sum as spark_sum, round as spark_round, avg

# Inisialisasi SparkSession
spark = SparkSession.builder \
    .appName("Tugas6_2505060072_Muhammad_Raka_Mushaddaq") \
    .master("local[*]") \
    .config("spark.sql.warehouse.dir", "/tmp/spark-warehouse") \
    .getOrCreate()

# Mengatur level log agar konsol tetap bersih dan informatif
spark.sparkContext.setLogLevel("ERROR")

print(f"SparkSession berhasil diinisialisasi!")
print(f"Versi Spark  : {spark.version}")
print(f"Nama Aplikasi: {spark.sparkContext.appName}")

SparkSession berhasil diinisialisasi!
Versi Spark  : 3.5.9
Nama Aplikasi: Tugas6_2505060072_Muhammad_Raka_Mushaddaq


---
## Bagian A. EXTRACT

**Instruksi:**
> Baca ketiga sumber data (`tugas6_transaksi.csv`, `tugas6_produk.json`, `tugas6_ulasan.csv`) menjadi tiga Spark DataFrame terpisah. Tampilkan jumlah baris dan `printSchema()` masing-masing.


In [5]:
# 1. Membaca Data Transaksi (CSV)
df_transaksi = spark.read.csv("tugas6_transaksi.csv", header=True, inferSchema=True)

# 2. Membaca Data Produk (JSON Lines)
df_produk = spark.read.json("tugas6_produk.json")

# 3. Membaca Data Ulasan Pelanggan (CSV)
df_ulasan = spark.read.csv("tugas6_ulasan.csv", header=True, inferSchema=True)

print("=" * 60)
print("HASIL TAHAP EXTRACT: JUMLAH BARIS & SKEMA")
print("=" * 60)

print(f"1. df_transaksi : {df_transaksi.count()} baris")
df_transaksi.printSchema()
df_transaksi.show(5)

print(f"2. df_produk    : {df_produk.count()} baris")
df_produk.printSchema()
df_produk.show(5)

print(f"3. df_ulasan    : {df_ulasan.count()} baris")
df_ulasan.printSchema()
df_ulasan.show(5)

HASIL TAHAP EXTRACT: JUMLAH BARIS & SKEMA
1. df_transaksi : 5000 baris
root
 |-- order_id: string (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)

+--------+----------+------------+-------------------+
|order_id|product_id|unit_terjual|            tanggal|
+--------+----------+------------+-------------------+
|     TX0|        21|           2|2026-10-10 00:00:00|
|     TX1|         8|           6|2026-10-25 00:00:00|
|     TX2|        25|           4|2026-10-13 00:00:00|
|     TX3|         3|           4|2026-10-18 00:00:00|
|     TX4|        19|           6|2026-10-07 00:00:00|
+--------+----------+------------+-------------------+
only showing top 5 rows

2. df_produk    : 30 baris
root
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- product_id: long (nullable = true)

+------+------------+-----------+---------

---
## Bagian B. TRANSFORM — Penggabungan

**Instruksi:**
> Gabungkan ketiga DataFrame menjadi satu (`order_id` sebagai kunci ke ulasan, `product_id` sebagai kunci ke produk).
> Gunakan **`salah satu join yang tepat`** dari transaksi ke ulasan (karena tidak semua transaksi memiliki ulasan) dan **`salah satu join yang tepat`** dari transaksi ke produk (karena setiap transaksi pasti memiliki produk yang valid).
> Tambahkan kolom `total_pendapatan` (`unit_terjual x harga`).


In [6]:
# Langkah 1: Melakukan penggabungan (Join) ketiga DataFrame
# - Transaksi ke Ulasan : LEFT JOIN (karena tidak semua transaksi memiliki ulasan, namun seluruh transaksi wajib dipertahankan)
# - Transaksi ke Produk : INNER JOIN (karena setiap transaksi pasti memiliki padanan data produk yang valid di katalog)

df_gabung = df_transaksi.join(df_ulasan, on="order_id", how="left") \
                        .join(df_produk, on="product_id", how="inner")

# Langkah 2: Menambahkan kolom kalkulasi total_pendapatan = unit_terjual * harga
df_gabung = df_gabung.withColumn("total_pendapatan", col("unit_terjual") * col("harga"))

print("=" * 60)
print(f"HASIL TAHAP TRANSFORM (PENGGABUNGAN): {df_gabung.count()} baris")
print("=" * 60)
df_gabung.printSchema()
df_gabung.show(10, truncate=False)

HASIL TAHAP TRANSFORM (PENGGABUNGAN): 5000 baris
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- rating: integer (nullable = true)
 |-- harga: long (nullable = true)
 |-- kategori: string (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- total_pendapatan: long (nullable = true)

+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|product_id|order_id|unit_terjual|tanggal            |rating|harga |kategori    |nama_produk|total_pendapatan|
+----------+--------+------------+-------------------+------+------+------------+-----------+----------------+
|21        |TX0     |2           |2026-10-10 00:00:00|1     |25000 |Elektronik  |Produk-21  |50000           |
|8         |TX1     |6           |2026-10-25 00:00:00|NULL  |250000|Fashion     |Produk-8   |1500000         |
|25        |TX2  

---
## Bagian C. TRANSFORM — Penanganan Data Kosong & Pengayaan

**Instruksi:**
> 1. Tambahkan kolom `ada_ulasan` bernilai `True`/`False` (tidak boleh diisi manual satu satu, **sebelum** langkah `na.fill()`).
> 2. Transaksi tanpa ulasan akan memiliki `rating` bernilai kosong (`null`) setelah `left join` — isi nilai kosong tersebut dengan angka **0** menggunakan `na.fill()`.
> 3. Sertakan alasan singkat mengapa 0 (bukan nilai lain) masuk akal untuk kasus "belum ada ulasan".


In [7]:
# Langkah 1: Menambahkan kolom boolean 'ada_ulasan' SEBELUM penanganan null
# Nilai True jika rating tidak null, False jika rating null
df_enriched = df_gabung.withColumn("ada_ulasan", col("rating").isNotNull())

print("Pengecekan sebelum na.fill():")
print(f"- Jumlah baris dengan rating NULL     : {df_enriched.filter(col('rating').isNull()).count()}")
print(f"- Jumlah baris dengan ada_ulasan=True : {df_enriched.filter(col('ada_ulasan') == True).count()}")
print(f"- Jumlah baris dengan ada_ulasan=False: {df_enriched.filter(col('ada_ulasan') == False).count()}")

# Langkah 2: Mengisi nilai null pada kolom rating dengan angka 0
df_bersih = df_enriched.na.fill(0, subset=["rating"])

print("\nPengecekan setelah na.fill(0):")
print(f"- Jumlah baris dengan rating NULL     : {df_bersih.filter(col('rating').isNull()).count()}")
print(f"- Jumlah baris dengan rating == 0     : {df_bersih.filter(col('rating') == 0).count()}")
print(f"- Total baris akhir                   : {df_bersih.count()}")

print("\n=== TAMPILAN 10 BARIS DATA HASIL TRANSFORM & DATA QUALITY ===")
df_bersih.select(
    "order_id", "nama_produk", "kategori", "unit_terjual", 
    "harga", "total_pendapatan", "rating", "ada_ulasan"
).show(10, truncate=False)

Pengecekan sebelum na.fill():
- Jumlah baris dengan rating NULL     : 1500
- Jumlah baris dengan ada_ulasan=True : 3500
- Jumlah baris dengan ada_ulasan=False: 1500

Pengecekan setelah na.fill(0):
- Jumlah baris dengan rating NULL     : 0
- Jumlah baris dengan rating == 0     : 1500
- Total baris akhir                   : 5000

=== TAMPILAN 10 BARIS DATA HASIL TRANSFORM & DATA QUALITY ===
+--------+-----------+------------+------------+------+----------------+------+----------+
|order_id|nama_produk|kategori    |unit_terjual|harga |total_pendapatan|rating|ada_ulasan|
+--------+-----------+------------+------------+------+----------------+------+----------+
|TX0     |Produk-21  |Elektronik  |2           |25000 |50000           |1     |true      |
|TX1     |Produk-8   |Fashion     |6           |250000|1500000         |0     |false     |
|TX2     |Produk-25  |Elektronik  |4           |25000 |100000          |2     |true      |
|TX3     |Produk-3   |Fashion     |4           |75000 |300000 

---
## Bagian D. LOAD 

**Instruksi:**
> Simpan hasil akhir ke HDFS dalam format **Parquet**, dipartisi berdasarkan `kategori`, ke path `/user/[username]/tugas6/hasil_etl`.
> Verifikasi dengan `hdfs dfs -ls -R`, lalu baca kembali dan tampilkan `count()`-nya sebagai bukti data tersimpan utuh.


In [11]:
# Menentukan path target penyimpanan di HDFS
# Menyimpan ke path HDFS: /user/mahasiswa/tugas6/hasil_etl (dan salinan ke /user/raka/tugas6/hasil_etl)
hdfs_target_path = "hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl"

# Menuliskan DataFrame akhir ke HDFS dalam format Parquet dengan partisi berdasarkan kolom 'kategori'
df_bersih.write.mode("overwrite").partitionBy("kategori").parquet(hdfs_target_path)

# Menyimpan juga ke direktori user raka untuk fleksibilitas pemeriksaan lingkungan
df_bersih.write.mode("overwrite").partitionBy("kategori").parquet("hdfs://localhost:9000/user/raka/tugas6/hasil_etl")

print(f"Data ETL berhasil disimpan ke HDFS: {hdfs_target_path}")

[Stage 89:>                                                         (0 + 1) / 1]

Data ETL berhasil disimpan ke HDFS: hdfs://localhost:9000/user/mahasiswa/tugas6/hasil_etl


In [12]:
# Verifikasi struktur folder dan berkas partisi di HDFS menggunakan perintah Hadoop shell
!hdfs dfs -ls -R /user/mahasiswa/tugas6/hasil_etl

-rw-r--r--   3 raka supergroup          0 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/_SUCCESS
drwxr-xr-x   - raka supergroup          0 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik
-rw-r--r--   3 raka supergroup      16366 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Elektronik/part-00000-92cda253-8f06-4dd2-9247-e07cb9744973.c000.snappy.parquet
drwxr-xr-x   - raka supergroup          0 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion
-rw-r--r--   3 raka supergroup      11133 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Fashion/part-00000-92cda253-8f06-4dd2-9247-e07cb9744973.c000.snappy.parquet
drwxr-xr-x   - raka supergroup          0 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan
-rw-r--r--   3 raka supergroup      10392 2026-09-27 15:38 /user/mahasiswa/tugas6/hasil_etl/kategori=Kesehatan/part-00000-92cda253-8f06-4dd2-9247-e07cb9744973.c000.snappy.parquet
drwxr-xr-x   - raka supergroup 

In [13]:
df_verifikasi_hdfs = spark.read.parquet(hdfs_target_path)

print("=" * 60)
print("VERIFIKASI HASIL BACA KEMBALI DARI HDFS")
print("=" * 60)
print(f"Jumlah baris tersimpan di HDFS : {df_verifikasi_hdfs.count()} baris")
print("Skema berkas Parquet dari HDFS :")
df_verifikasi_hdfs.printSchema()

print("\nCuplikan 5 baris data dari HDFS:")
df_verifikasi_hdfs.show(5)

VERIFIKASI HASIL BACA KEMBALI DARI HDFS
Jumlah baris tersimpan di HDFS : 5000 baris
Skema berkas Parquet dari HDFS :
root
 |-- product_id: integer (nullable = true)
 |-- order_id: string (nullable = true)
 |-- unit_terjual: integer (nullable = true)
 |-- tanggal: timestamp (nullable = true)
 |-- rating: integer (nullable = true)
 |-- harga: long (nullable = true)
 |-- nama_produk: string (nullable = true)
 |-- total_pendapatan: long (nullable = true)
 |-- ada_ulasan: boolean (nullable = true)
 |-- kategori: string (nullable = true)


Cuplikan 5 baris data dari HDFS:
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|product_id|order_id|unit_terjual|            tanggal|rating| harga|nama_produk|total_pendapatan|ada_ulasan|  kategori|
+----------+--------+------------+-------------------+------+------+-----------+----------------+----------+----------+
|        21|     TX0|           2|2026-10-10 00:00:00|     1| 25000

---
## Bagian E. Insight Akhir

**Instruksi:**
> Dari data hasil ETL, tampilkan (menggunakan DataFrame API **atau** Spark SQL, bebas memilih): kategori produk mana yang memiliki **persentase transaksi dengan ulasan** (`ada_ulasan = True`) **paling rendah**?
> Tulis 2-3 kalimat interpretasi bisnis pada markdown cell: mengapa hal ini mungkin penting diketahui oleh tim marketing?


In [15]:
# Pendekatan 1: Menggunakan DataFrame API
insight_df_api = df_verifikasi_hdfs.groupBy("kategori").agg(
    count("order_id").alias("total_transaksi"),
    spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)).alias("transaksi_dengan_ulasan"),
    spark_round(
        (spark_sum(when(col("ada_ulasan") == True, 1).otherwise(0)) / count("order_id")) * 100, 2
    ).alias("persentase_berulasan")
).orderBy(col("persentase_berulasan").asc())

print("=== INSIGHT 1: MENGGUNAKAN DATAFRAME API ===")
insight_df_api.show(truncate=False)

=== INSIGHT 1: MENGGUNAKAN DATAFRAME API ===
+------------+---------------+-----------------------+--------------------+
|kategori    |total_transaksi|transaksi_dengan_ulasan|persentase_berulasan|
+------------+---------------+-----------------------+--------------------+
|Makanan     |536            |369                    |68.84               |
|Kesehatan   |961            |662                    |68.89               |
|Fashion     |1035           |726                    |70.14               |
|Rumah Tangga|815            |575                    |70.55               |
|Elektronik  |1653           |1168                   |70.66               |
+------------+---------------+-----------------------+--------------------+



In [17]:
# Pendekatan 2: Menggunakan Spark SQL (sebagai validasi komplementer)
df_verifikasi_hdfs.createOrReplaceTempView("hasil_etl_ecommerce")

insight_sql = spark.sql("""
    SELECT 
        kategori,
        COUNT(order_id) AS total_transaksi,
        SUM(CASE WHEN ada_ulasan = true THEN 1 ELSE 0 END) AS transaksi_dengan_ulasan,
        ROUND((SUM(CASE WHEN ada_ulasan = true THEN 1 ELSE 0 END) * 100.0) / COUNT(order_id), 2) AS persentase_berulasan
    FROM hasil_etl_ecommerce
    GROUP BY kategori
    ORDER BY persentase_berulasan ASC
""")

print("=== INSIGHT 2: MENGGUNAKAN SPARK SQL ===")
insight_sql.show(truncate=False)

=== INSIGHT 2: MENGGUNAKAN SPARK SQL ===
+------------+---------------+-----------------------+--------------------+
|kategori    |total_transaksi|transaksi_dengan_ulasan|persentase_berulasan|
+------------+---------------+-----------------------+--------------------+
|Makanan     |536            |369                    |68.84               |
|Kesehatan   |961            |662                    |68.89               |
|Fashion     |1035           |726                    |70.14               |
|Rumah Tangga|815            |575                    |70.55               |
|Elektronik  |1653           |1168                   |70.66               |
+------------+---------------+-----------------------+--------------------+



---
## Menutup SparkSession

Menutup sesi Spark secara aman untuk melepaskan alokasi memori JVM dan sumber daya komputasi kluster.


In [18]:
# Menutup SparkSession
spark.stop()
print("SparkSession berhasil ditutup secara aman.")

SparkSession berhasil ditutup secara aman.
